# Image Quality: CNN + pretrained ResNet18 (GPU training)

**Prerequisites:** Put `selected_2000.zip` and `portfolio_projects_full_implementation.zip` in `My Drive/portfolio_inputs/` (create folder first). In Google Colab, select **Runtime → Change runtime type → T4 GPU** if available. Running will generate 20,000 variants (may take time), train both networks, write held-out metrics, and copy models into `My Drive/portfolio_results/`. No scores are claimed before training finishes. Python code comes from the supplied project package; confirm it matches the reviewed source.

**Important:** The four synthetic classes measure the dataset generator's corruption recipes, not independently observed image-quality defects. This notebook uses pretrained ImageNet weights for ResNet18 and does not touch test data for selection (best checkpoint is chosen by validation macro F1).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
inputs=Path('/content/drive/MyDrive/portfolio_inputs')
assert (inputs/'selected_2000.zip').is_file(), 'Upload selected_2000.zip to My Drive/portfolio_inputs/'
assert (inputs/'portfolio_projects_full_implementation.zip').is_file(), 'Upload portfolio_projects_full_implementation.zip to My Drive/portfolio_inputs/'
import torch
print('CUDA available:',torch.cuda.is_available())
assert torch.cuda.is_available(), 'Select a GPU runtime before training'


In [ ]:
import zipfile,shutil,os
work=Path('/content/portfolio_work');work.mkdir(exist_ok=True)
with zipfile.ZipFile(inputs/'portfolio_projects_full_implementation.zip') as z: z.extractall(work)
with zipfile.ZipFile(inputs/'selected_2000.zip') as z: z.extractall(work/'photos')
project=work/'portfolio_projects_full'/'02_image_quality'
assert (project/'quality/train.py').is_file()
photos=work/'photos'
print('Project:',project,'Images:',len(list(photos.rglob('*.jpg'))))


In [ ]:
%pip -q install -r /content/portfolio_work/portfolio_projects_full/02_image_quality/requirements.txt

In [ ]:
import subprocess,sys,json
out=work/'generated'
if not (out/'build_report.json').exists():
 subprocess.run([sys.executable,'-m','quality.data','--sources',str(photos),'--output',str(out),'--variants','10','--size','128'],check=True,cwd=project)
report=json.loads((out/'build_report.json').read_text())
assert report['valid_sources']==2000 and report['variants']==20000, report
print(report)


In [ ]:
# Train custom CNN using source-grouped manifest, checkpoint selected on validation F1.
subprocess.run([sys.executable,'-m','quality.train','--model','cnn','--data',str(out),'--out',str(work/'models/cnn'),'--epochs','8','--size','128','--batch','64'],check=True,cwd=project)
print(json.loads((work/'models/cnn/metadata.json').read_text())['metrics']['test']['macro_f1'])


In [ ]:
# Genuine ImageNet transfer learning; requires a successful weight download.
subprocess.run([sys.executable,'-m','quality.train','--model','transfer','--pretrained','--data',str(out),'--out',str(work/'models/transfer'),'--epochs','8','--size','128','--batch','64'],check=True,cwd=project)
print(json.loads((work/'models/transfer/metadata.json').read_text())['metrics']['test']['macro_f1'])


In [ ]:
# Export artifacts and metadata into persistent Drive storage.
destination=Path('/content/drive/MyDrive/portfolio_results');destination.mkdir(exist_ok=True)
for kind in ('cnn','transfer'):
 src=work/'models'/kind
 if not (src/'metadata.json').exists():
  print(kind,'not completed; nothing copied');continue
 shutil.copytree(src,destination/kind,dirs_exist_ok=True)
 print(kind,'saved to',destination/kind)


## Evidence limitations
- Only claim CNN and transfer-learning metrics if their `metadata.json` files are produced and all files are exported.
- The OpenCV full-scale macro F1 (0.9289) was measured in a separate CPU environment; compare methods by their held-out test protocol, not CPU/GPU timing.
- The independent image collection and full 45-run benchmark remain separate experiments.
- If Colab disconnects, saved Drive artifacts survive only **after the export cell runs**. Consider saving per-model immediately for long runs.